In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

### bar analysis_NMF

In [6]:

# NMF_0519_MovieLens_bar_figures.py
# Bar figures for MovieLens NMF recommendations
# Decade only; ORIGINAL is always first bar

import os
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# PATHS
# =========================

DATA_DIR = Path("/home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/data")

OUT_BASE = Path("/home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/0519_NMF_bar_figures")
OUT_BASE.mkdir(parents=True, exist_ok=True)

# =========================
# SETTINGS
# =========================

MODEL_NAME = "NMF"

TARGET_DECADES = {
    "1920", "1930", "1940", "1950",
    "1960", "1970", "1980", "1990"
}

K_LIST = [15, 25, 35]

# =========================
# STORAGE
# =========================

results = defaultdict(lambda: defaultdict(dict))

# =========================
# PARSE FILENAME
# =========================

def parse_filename(fname):
    """
    Expected files:
      ORIGINAL_15recommendation.csv
      ORIGINAL_25recommendation.csv
      ORIGINAL_35recommendation.csv

      f_1920_2u_pos5_neg1_movies_15recommendation.csv
      f_1920_6u_pos5_neg1_movies_25recommendation.csv
      etc.
    """

    original_match = re.match(
        r"^ORIGINAL_(\d+)recommendation\.csv$",
        fname
    )

    if original_match:
        k = int(original_match.group(1))
        return "ORIGINAL", "original", k

    injection_match = re.match(
        r"^f_(\d{4})_(\d+)u_pos(\d+)_neg(NA|0|1|\d+)_movies_(\d+)recommendation\.csv$",
        fname
    )

    if injection_match:
        decade = injection_match.group(1)
        n = f"{injection_match.group(2)}u"
        pos_rating = int(injection_match.group(3))
        k = int(injection_match.group(5))

        if pos_rating != 5:
            return None, None, None

        return decade, n, k

    return None, None, None

# =========================
# METRIC
# =========================

def compute_avg(df, target_decade):
    df = df.copy()

    if "decade" not in df.columns:
        raise ValueError("Missing required column: decade")

    df["decade"] = df["decade"].astype(str).str.replace(r"\.0$", "", regex=True)

    df["has_decade"] = (
        df["decade"] == str(target_decade)
    )

    user_counts = df.groupby("user_id")["has_decade"].sum()

    return user_counts.mean(), len(user_counts)

# =========================
# READ ALL FILES
# =========================

all_n_values = set()

for fname in os.listdir(DATA_DIR):

    if not fname.endswith(".csv"):
        continue

    decade, n, k = parse_filename(fname)

    if decade is None or k not in K_LIST:
        continue

    try:
        df = pd.read_csv(DATA_DIR / fname)

        if n == "original":
            # ORIGINAL is evaluated separately for every target decade
            for target_decade in TARGET_DECADES:
                avg_val, num_users = compute_avg(df, target_decade)

                results[target_decade][k]["original"] = (
                    avg_val,
                    num_users
                )

            all_n_values.add("original")

        else:
            avg_val, num_users = compute_avg(df, decade)

            results[decade][k][n] = (
                avg_val,
                num_users
            )

            all_n_values.add(n)

    except Exception as e:
        print(f"Error reading {fname}: {e}")

# =========================
# COLORS
# =========================

def generate_colors(n_list):
    palette = [
        "#4E79A7", "#59A14F", "#E15759", "#B07AA1",
        "#F28E2B", "#76B7B2", "#EDC948", "#9C755F",
        "#FF9DA7", "#8CD17D", "#86BCB6", "#BAB0AC"
    ]

    return {
        n: palette[i % len(palette)]
        for i, n in enumerate(n_list)
    }

# =========================
# ORDERING
# =========================

def n_sort_key(n):
    if n == "original":
        return -1
    return int(n.replace("u", ""))

# =========================
# PLOTTING
# =========================

for decade in sorted(results.keys(), key=lambda x: int(x)):

    k_data = results[decade]
    ks = sorted(k_data.keys())

    n_types = sorted(
        set(n for k in ks for n in k_data[k].keys()),
        key=n_sort_key
    )

    # Guarantee ORIGINAL is first bar
    if "original" in n_types:
        n_types = ["original"] + [n for n in n_types if n != "original"]

    color_map = generate_colors(n_types)

    x = np.arange(len(ks)) * 1.2
    width = 1.0 / len(n_types)

    fig, ax = plt.subplots(figsize=(18, 11))

    max_val = 0
    bar_containers = []

    for i, n in enumerate(n_types):

        vals = [
            k_data[k].get(n, (0, 0))[0]
            for k in ks
        ]

        label_name = "ORIGINAL" if n == "original" else n

        bars = ax.bar(
            x + (i - len(n_types) / 2) * width + width / 2,
            vals,
            width=width,
            color=color_map[n],
            edgecolor="black",
            linewidth=0.6,
            label=label_name
        )

        bar_containers.append((bars, vals))
        max_val = max(max_val, max(vals) if vals else 0)

    # =========================
    # STYLE
    # =========================

    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)

    ax.yaxis.grid(
        True,
        linestyle="--",
        linewidth=0.6,
        alpha=0.6
    )

    ax.set_title(
        f"{MODEL_NAME} - Decade {decade}",
        fontsize=40,
        fontweight="bold",
        pad=20
    )

    ax.set_xlabel(
        "k (recommendation size)",
        fontsize=38,
        labelpad=14
    )

    ax.set_ylabel(
        "Avg count per user",
        fontsize=38,
        labelpad=14
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        ks,
        fontsize=34
    )

    ax.tick_params(
        axis="y",
        labelsize=34
    )

    ax.legend(
        fontsize=22,
        ncol=3
    )

    # =========================
    # BAR LABELS
    # =========================

    if max_val == 0:
        max_val = 1

    for bars, vals in bar_containers:

        for bar, val in zip(bars, vals):

            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + max_val * 0.015,
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=22,
                rotation=90,
            )

    ax.set_ylim(top=max_val * 1.25)

    # =========================
    # SAVE FIGURE
    # =========================

    decade_folder = OUT_BASE / f"nmf_decade_{decade}_analysis"

    decade_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    fig_path = decade_folder / f"NMF - Decade {decade}.png"

    plt.tight_layout()

    plt.savefig(
        fig_path,
        dpi=300
    )

    plt.close()

    # =========================
    # SAVE EXPLANATION TEXT
    # =========================

    explanation_path = decade_folder / f"NMF - Decade {decade}.txt"

    with open(explanation_path, "w") as f:

        f.write(
            f"NMF Analysis for Decade: {decade}\n\n"
        )

        f.write(
            "Metric: Average count of target decade per user\n\n"
        )

        for k in ks:

            f.write(f"k = {k}\n")

            for n in n_types:

                val, users = k_data[k].get(n, (0, 0))
                label_name = "ORIGINAL" if n == "original" else n

                f.write(
                    f"  {label_name}: {val:.4f} (users={users})\n"
                )

            f.write("\n")

print("All NMF figures and explanation files saved.")



All NMF figures and explanation files saved.


### param

In [7]:
#!/usr/bin/env python3
# NMF_0519_MovieLens_details_figures.py
# Detail figures for MovieLens NMF results
# Decade only

import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# ── Paths ──
BASE = "/home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result"

BI_DIR = os.path.join(BASE, "bi")
MU_DIR = os.path.join(BASE, "mio")   # using mio for MU
BU_DIR = os.path.join(BASE, "bu")

# ── Root figures folder ──
DETAILS_FIG_DIR = os.path.join(
    BASE,
    "0519_NMF_details_figures"
)

os.makedirs(
    DETAILS_FIG_DIR,
    exist_ok=True
)

# ── Patterns ──
BI_PATTERN = re.compile(
    r"^f_(.+?)_(\d+)u_pos5_neg1_movies_bi_per_decade\.csv$"
)

# ── Step 1: Discover bi files ──
bi_files = {}

for fname in os.listdir(BI_DIR):

    m = BI_PATTERN.match(fname)

    if m:

        decade_key = m.group(1)
        n = int(m.group(2))

        bi_files[(decade_key, n)] = fname

print(f"Found {len(bi_files)} bi files")

# ── Step 2: Load and merge in memory only ──

merged_data = {}
bu_data = {}

for (decade_key, n), bi_fname in sorted(bi_files.items()):

    mu_fname = bi_fname.replace(
        "_bi_",
        "_mu_"
    )

    bu_fname = (
        bi_fname
        .replace("_bi_", "_bu_")
        .replace(
            "_per_decade.csv",
            "_per_decade_biased.csv"
        )
    )

    bi_path = os.path.join(BI_DIR, bi_fname)
    mu_path = os.path.join(MU_DIR, mu_fname)
    bu_path = os.path.join(BU_DIR, bu_fname)

    if not (
        os.path.exists(mu_path)
        and os.path.exists(bu_path)
    ):

        print(
            f"WARNING: Missing mu/bu for {bi_fname}, skipping."
        )

        print(f"  Expected mu: {mu_fname}")
        print(f"  Expected bu: {bu_fname}")

        continue

    df_bi = pd.read_csv(bi_path)
    df_mu = pd.read_csv(mu_path)
    df_bu = pd.read_csv(bu_path)

    # clean decade column
    for df_name, df in [
        ("bi", df_bi),
        ("mu", df_mu),
        ("bu", df_bu)
    ]:

        if "decade" in df.columns:

            df["decade"] = (
                df["decade"]
                .astype(str)
                .str.strip()
            )

            df = df[
                df["decade"].str.lower() != "unknown"
            ].copy()

        if df_name == "bi":
            df_bi = df

        elif df_name == "mu":
            df_mu = df

        else:
            df_bu = df

    bu_data[(decade_key, n)] = df_bu.copy()

    # keep only non-biasing users for main merge
    if "user_type" in df_bu.columns:

        df_bu_merge = df_bu[
            df_bu["user_type"] == "non_biasing"
        ].copy()

        if df_bu_merge.empty:
            df_bu_merge = df_bu.copy()

    else:
        df_bu_merge = df_bu.copy()

    merge_cols = ["decade"]

    if "is_target" in df_bi.columns:
        merge_cols.append("is_target")

    df = pd.merge(
        df_mu,
        df_bi,
        on=merge_cols,
        how="outer"
    )

    df = pd.merge(
        df,
        df_bu_merge,
        on=merge_cols,
        how="outer"
    )

    merged_data[(decade_key, n)] = df

print(f"Merged {len(merged_data)} configs")

# ── Step 3: Organize by decade ──

decade_groups = defaultdict(dict)

for (decade_key, n), df in merged_data.items():

    decade_groups[decade_key][n] = df

bu_groups = defaultdict(dict)

for (decade_key, n), df in bu_data.items():

    bu_groups[decade_key][n] = df

# ── Style ──

plt.style.use("seaborn-v0_8-pastel")

TARGET_COLOR = "#5B3A29"

NON_TARGET_LINEWIDTH = 1.5
TARGET_LINEWIDTH = 3.5

PASTEL_COLORS = [
    "#AEC6CF",
    "#FFD1DC",
    "#B5EAD7",
    "#FFDAC1",
    "#C3B1E1",
    "#F3E5AB",
    "#FFB7B2",
    "#B0E0E6",
    "#D5AAFF",
    "#FFFACD",
    "#E0BBE4",
    "#957DAD",
    "#D4E157",
    "#80CBC4",
    "#F48FB1",
]

TARGET_BIAS_COLOR = "#8B0000"
TARGET_NONBIAS_COLOR = "#5B3A29"

NONTARGET_BIAS_COLOR = "#1F77B4"
NONTARGET_NONBIAS_COLOR = "#7FB3D5"

# ── Helpers ──

def get_single_value(row, col):

    if row.empty or col not in row.columns:
        return None

    return row[col].values[0]


def get_decade_output_dir(decade_key):

    decade_dir = os.path.join(
        DETAILS_FIG_DIR,
        str(decade_key)
    )

    os.makedirs(
        decade_dir,
        exist_ok=True
    )

    return decade_dir


def plot_figure(
    decade_key,
    sorted_ns,
    n_dict,
    target_d,
    metric_name,
    compute_value,
    ylabel,
    suffix
):

    all_decades = set()

    for n in sorted_ns:

        if "decade" in n_dict[n].columns:

            all_decades.update(
                n_dict[n]["decade"]
                .dropna()
                .tolist()
            )

    all_decades = sorted(all_decades)

    decade_series = {}

    for d in all_decades:

        pts = []

        for n in sorted_ns:

            df = n_dict[n]

            row = df[df["decade"] == d]

            if not row.empty:

                val = compute_value(row)

                if val is not None:
                    pts.append((n, val))

        decade_series[d] = pts

    non_target = [
        g for g in all_decades
        if g != target_d
    ]

    color_map = {}

    for i, g in enumerate(non_target):

        color_map[g] = (
            PASTEL_COLORS[
                i % len(PASTEL_COLORS)
            ]
        )

    color_map[target_d] = TARGET_COLOR

    positions = list(range(len(sorted_ns)))

    n_to_pos = {
        n: p
        for n, p in zip(sorted_ns, positions)
    }

    fig, ax = plt.subplots(figsize=(14, 8))

    # non-target lines
    for g in non_target:

        pts = decade_series[g]

        if pts:

            xs = [
                n_to_pos[x]
                for x, _ in pts
            ]

            ys = [
                y
                for _, y in pts
            ]

            ax.plot(
                xs,
                ys,
                marker="o",
                color=color_map[g],
                linewidth=NON_TARGET_LINEWIDTH,
                markersize=9,
                alpha=0.8,
                label=g
            )

    # target line
    pts = decade_series.get(target_d, [])

    if pts:

        xs = [
            n_to_pos[x]
            for x, _ in pts
        ]

        ys = [
            y
            for _, y in pts
        ]

        ax.plot(
            xs,
            ys,
            marker="o",
            color=TARGET_COLOR,
            linewidth=TARGET_LINEWIDTH,
            markersize=7,
            zorder=10,
            label=f"{target_d} (target)"
        )

    ax.set_title(
        f"NMF - Decade {decade_key} ({metric_name})",
        fontsize=26,
        fontweight="bold"
    )

    ax.set_xlabel(
        "Number of Fictitious Users (n)",
        fontsize=25
    )

    ax.set_ylabel(
        ylabel,
        fontsize=28,
        fontweight="bold"
    )

    ax.set_xticks(positions)

    ax.set_xticklabels(
        [str(n) for n in sorted_ns],
        rotation=45
    )

    ax.legend(
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        fontsize=10
    )

    ax.grid(True, alpha=0.3)

    plt.tight_layout()

    decade_dir = get_decade_output_dir(decade_key)

    out = os.path.join(
        decade_dir,
        f"{suffix}.png"
    )

    fig.savefig(
        out,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(f"Saved: {out}")


def plot_bu_bias_comparison(
    decade_key,
    sorted_ns,
    bu_n_dict,
    ylabel,
    suffix
):

    positions = list(range(len(sorted_ns)))

    n_to_pos = {
        n: p
        for n, p in zip(sorted_ns, positions)
    }

    target_bias_pts = []
    target_nonbias_pts = []

    non_target_bias_pts = []
    non_target_nonbias_pts = []

    for n in sorted_ns:

        df = bu_n_dict[n].copy()

        if "decade" in df.columns:

            df["decade"] = (
                df["decade"]
                .astype(str)
                .str.strip()
            )

            df = df[
                df["decade"].str.lower() != "unknown"
            ]

        if "user_type" in df.columns:

            df["user_type"] = (
                df["user_type"]
                .astype(str)
                .str.strip()
            )

        if "is_target" in df.columns:

            df["is_target"] = (
                df["is_target"]
                .astype(str)
                .str.strip()
            )

        row = df[
            (df["is_target"] == "target")
            & (df["user_type"] == "biasing")
        ]

        if not row.empty:
            target_bias_pts.append(
                (n, row["avg_bu"].mean())
            )

        row = df[
            (df["is_target"] == "target")
            & (df["user_type"] == "non_biasing")
        ]

        if not row.empty:
            target_nonbias_pts.append(
                (n, row["avg_bu"].mean())
            )

        row = df[
            (df["is_target"] == "non_target")
            & (df["user_type"] == "biasing")
        ]

        if not row.empty:
            non_target_bias_pts.append(
                (n, row["avg_bu"].mean())
            )

        row = df[
            (df["is_target"] == "non_target")
            & (df["user_type"] == "non_biasing")
        ]

        if not row.empty:
            non_target_nonbias_pts.append(
                (n, row["avg_bu"].mean())
            )

    fig, ax = plt.subplots(figsize=(14, 8))

    series_info = [
        (
            "Target - biasing",
            target_bias_pts,
            TARGET_BIAS_COLOR,
            3.5,
            "o"
        ),
        (
            "Target - non_biasing",
            target_nonbias_pts,
            TARGET_NONBIAS_COLOR,
            3.5,
            "s"
        ),
        (
            "Non-target - biasing",
            non_target_bias_pts,
            NONTARGET_BIAS_COLOR,
            2.5,
            "^"
        ),
        (
            "Non-target - non_biasing",
            non_target_nonbias_pts,
            NONTARGET_NONBIAS_COLOR,
            2.5,
            "D"
        ),
    ]

    for label, pts, color, linewidth, marker in series_info:

        if pts:

            xs = [
                n_to_pos[x]
                for x, _ in pts
            ]

            ys = [
                y
                for _, y in pts
            ]

            ax.plot(
                xs,
                ys,
                marker=marker,
                color=color,
                linewidth=linewidth,
                markersize=8,
                label=label
            )

    ax.set_title(
        f"NMF - Decade {decade_key} (bᵤ biased vs unbiased)",
        fontsize=26,
        fontweight="bold"
    )

    ax.set_xlabel(
        "Number of Fictitious Users (n)",
        fontsize=25
    )

    ax.set_ylabel(
        ylabel,
        fontsize=28,
        fontweight="bold"
    )

    ax.set_xticks(positions)

    ax.set_xticklabels(
        [str(n) for n in sorted_ns],
        rotation=45
    )

    ax.legend(
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        fontsize=11
    )

    ax.grid(True, alpha=0.3)

    plt.tight_layout()

    decade_dir = get_decade_output_dir(decade_key)

    out = os.path.join(
        decade_dir,
        f"{suffix}.png"
    )

    fig.savefig(
        out,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(f"Saved: {out}")

# ── Step 4: Generate ALL figures ──

for decade_key, n_dict in sorted(decade_groups.items()):

    sorted_ns = sorted(n_dict.keys())

    if not sorted_ns:
        continue

    sample_df = n_dict[sorted_ns[0]]

    if "is_target" not in sample_df.columns:
        continue

    target_rows = sample_df[
        sample_df["is_target"] == "target"
    ]

    if target_rows.empty:
        continue

    target_d = target_rows["decade"].iloc[0]

    # μ + bᵢ
    plot_figure(
        decade_key,
        sorted_ns,
        n_dict,
        target_d,
        "μ + bᵢ",
        lambda r: (
            get_single_value(r, "global_mean_mu")
            + get_single_value(r, "avg_bi")
            if get_single_value(r, "global_mean_mu") is not None
            and get_single_value(r, "avg_bi") is not None
            else None
        ),
        r"$\mu + b_i$",
        "mu_plus_bi"
    )

    # bᵢ
    plot_figure(
        decade_key,
        sorted_ns,
        n_dict,
        target_d,
        "bᵢ",
        lambda r: get_single_value(r, "avg_bi"),
        r"$b_i$",
        "bi_only"
    )

    # bᵤ
    plot_figure(
        decade_key,
        sorted_ns,
        n_dict,
        target_d,
        "bᵤ (non-biasing users)",
        lambda r: get_single_value(r, "avg_bu"),
        r"$b_u$",
        "bu_only_non_biasing"
    )

    # μ
    plot_figure(
        decade_key,
        sorted_ns,
        n_dict,
        target_d,
        "μ",
        lambda r: get_single_value(r, "global_mean_mu"),
        r"$\mu$",
        "mu_only"
    )

    # biased vs unbiased comparison
    if decade_key in bu_groups:

        plot_bu_bias_comparison(
            decade_key,
            sorted_ns,
            bu_groups[decade_key],
            r"$b_u$",
            "bu_biased_vs_unbiased_target_vs_nontarget"
        )

print("Done! NMF figures saved per decade folder.")

Found 32 bi files
Merged 32 configs
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/0519_NMF_details_figures/1920/mu_plus_bi.png
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/0519_NMF_details_figures/1920/bi_only.png
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/0519_NMF_details_figures/1920/bu_only_non_biasing.png
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/0519_NMF_details_figures/1920/mu_only.png
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/0519_NMF_details_figures/1920/bu_biased_vs_unbiased_target_vs_nontarget.png
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/0519_NMF_details_figures/1930/mu_plus_bi.png
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/result/0519_NMF_details_figures/1930/bi_only.png
Saved: /home/moshtasa/Research/phd-recsys/Movie /current/0519/NMF_0519/r